[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_03/MFM_ch3_bvb_factor_sorts/MFM_ch3_bvb_factor_sorts.ipynb)

# MFM_ch3_bvb_factor_sorts

Seminar 3, Part C (reference analysis): monthly momentum (12-1) and low-beta long-short portfolios built from Bucharest Stock Exchange blue chips, with Newey-West t-statistics.

*Modelling Financial Markets (MFM), Chapter 3: Factor Models and Asset Pricing. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/MFM/Chapter_3'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import os
import io
import zipfile
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Data

Market prices from the course repository (`data/market`); factor and portfolio returns from the Kenneth French Data Library.

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
# copia locala a folderului data/market, daca notebook-ul ruleaza din repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
FRENCH = 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/'

SECTORS = ['XLB', 'XLE', 'XLF', 'XLI', 'XLK', 'XLP', 'XLU', 'XLV', 'XLY']        # din dec. 1998
SECTORS_ALL = SECTORS + ['XLRE', 'XLC']                                           # XLRE 2015, XLC 2018
SECTOR_NAMES = {'XLB': 'Materials', 'XLE': 'Energy', 'XLF': 'Financials', 'XLI': 'Industrials',
                'XLK': 'Technology', 'XLP': 'Cons. staples', 'XLU': 'Utilities', 'XLV': 'Health care',
                'XLY': 'Cons. discretionary', 'XLRE': 'Real estate', 'XLC': 'Communication'}
FACTOR_ETFS = ['MTUM', 'VLUE', 'QUAL', 'USMV', 'IWM', 'IWD', 'IWF']
BVB = ['TLV', 'SNP', 'BRD', 'TGN', 'FP', 'SNG', 'H2O', 'SNN', 'EL', 'DIGI', 'TEL']
BVB_NAMES = {'TLV': 'Banca Transilvania', 'SNP': 'OMV Petrom', 'BRD': 'BRD-GSG', 'TGN': 'Transgaz',
             'FP': 'Fondul Proprietatea', 'SNG': 'Romgaz', 'H2O': 'Hidroelectrica', 'SNN': 'Nuclearelectrica',
             'EL': 'Electrica', 'DIGI': 'Digi', 'TEL': 'Transelectrica'}

_CACHE = {}


def read_market(symbol):
    """Citeste data/market/<SIMBOL>.csv local sau din repo-ul GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname)
    src = path if os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def price(symbol):
    """Pretul folosit: adjusted_close pentru ETF/actiuni (.US, .RO), close pentru indici."""
    d = read_market(symbol)
    col = 'close' if symbol.endswith('.INDX') or symbol == 'BET' else 'adjusted_close'
    s = d[col].astype(float)
    return s[s > 0].rename(symbol)


def prices(symbols, start=None, end=None):
    """Preturi aliniate: join pe zilele comune (analiza comuna -> intai join pe preturi)."""
    p = pd.concat([price(s) for s in symbols], axis=1).dropna()
    return p.loc[start:end]


def log_returns(p):
    """Randamente log pe un tabel deja aliniat (zile comune)."""
    return np.log(p).diff().dropna()


# =============================================================================
# KENNETH FRENCH DATA LIBRARY)
# =============================================================================
FILES = {
    ('ff3', 'M'): 'F-F_Research_Data_Factors_CSV.zip',
    ('ff3', 'D'): 'F-F_Research_Data_Factors_daily_CSV.zip',
    ('ff5', 'M'): 'F-F_Research_Data_5_Factors_2x3_CSV.zip',
    ('ff5', 'D'): 'F-F_Research_Data_5_Factors_2x3_daily_CSV.zip',
    ('mom', 'M'): 'F-F_Momentum_Factor_CSV.zip',
    ('mom', 'D'): 'F-F_Momentum_Factor_daily_CSV.zip',
    ('p25', 'M'): '25_Portfolios_5x5_CSV.zip',
    ('ind10', 'M'): '10_Industry_Portfolios_CSV.zip',
    ('ind30', 'M'): '30_Industry_Portfolios_CSV.zip',
    ('p100', 'M'): '100_Portfolios_10x10_CSV.zip',
    # portofolii univariate (primul tabel: ponderate cu valoarea de piata), studiul de caz Jensen-Kelly-Pedersen
    ('ME', 'M'): 'Portfolios_Formed_on_ME_CSV.zip',
    ('BE-ME', 'M'): 'Portfolios_Formed_on_BE-ME_CSV.zip',
    ('OP', 'M'): 'Portfolios_Formed_on_OP_CSV.zip',
    ('INV', 'M'): 'Portfolios_Formed_on_INV_CSV.zip',
    ('E-P', 'M'): 'Portfolios_Formed_on_E-P_CSV.zip',
    ('CF-P', 'M'): 'Portfolios_Formed_on_CF-P_CSV.zip',
    ('D-P', 'M'): 'Portfolios_Formed_on_D-P_CSV.zip',
    ('AC', 'M'): 'Portfolios_Formed_on_AC_CSV.zip',
    ('NI', 'M'): 'Portfolios_Formed_on_NI_CSV.zip',
    ('BETA', 'M'): 'Portfolios_Formed_on_BETA_CSV.zip',
    ('VAR', 'M'): 'Portfolios_Formed_on_VAR_CSV.zip',
    ('RESVAR', 'M'): 'Portfolios_Formed_on_RESVAR_CSV.zip',
    ('PRIOR_12_2', 'M'): '10_Portfolios_Prior_12_2_CSV.zip',
    ('PRIOR_1_0', 'M'): '10_Portfolios_Prior_1_0_CSV.zip',
    ('PRIOR_60_13', 'M'): '10_Portfolios_Prior_60_13_CSV.zip',
}


def _parse_first_table(text, date_len):
    """Primul tabel din fisierul French: antet ',col1,col2...', apoi randuri YYYYMM(DD),valori."""
    lines = text.splitlines()
    i = next(k for k, l in enumerate(lines) if l.startswith(',') and len(l.split(',')) > 1)
    cols = [c.strip() for c in lines[i].split(',')[1:]]
    rows = []
    for l in lines[i + 1:]:
        parts = [x.strip() for x in l.split(',')]
        if not parts or len(parts[0]) != date_len or not parts[0].isdigit():
            break
        rows.append([parts[0]] + [float(x) for x in parts[1:]])
    df = pd.DataFrame(rows, columns=['date'] + cols)
    fmt = '%Y%m' if date_len == 6 else '%Y%m%d'
    df['date'] = pd.to_datetime(df['date'], format=fmt)
    if date_len == 6:
        df['date'] = df['date'] + pd.offsets.MonthEnd(0)
    df = df.set_index('date')
    return df.replace([-99.99, -999.0], np.nan) / 100.0          # procente -> zecimal


def french(name='ff5', freq='M'):
    """Factori Fama-French / momentum / portofolii, in zecimal (0.01 = 1%)."""
    key = (name, freq)
    if key in _CACHE:
        return _CACHE[key]
    url = FRENCH + FILES[key]
    raw = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'}),
                                 timeout=120).read()
    z = zipfile.ZipFile(io.BytesIO(raw))
    text = z.read(z.namelist()[0]).decode('latin-1')
    df = _parse_first_table(text, 6 if freq == 'M' else 8)
    df.columns = [c.replace('Mom', 'MOM').strip() for c in df.columns]
    _CACHE[key] = df
    return df


def factors(freq='M'):
    """Mkt-RF, SMB, HML, RMW, CMA, RF, MOM si SMB_FF3 pe perioada comuna.

    SMB din fisierul cu cinci factori (sortari dupa B/M, profitabilitate si investitii) intra in FF5;
    SMB_FF3 este SMB-ul publicat al modelului cu trei factori (sortari 2x3 dupa marime si B/M),
    folosit in FF3 si Carhart. HML, Mkt-RF si RF sunt aceleasi in cele doua fisiere.
    """
    f3 = french('ff3', freq)[['SMB']].rename(columns={'SMB': 'SMB_FF3'})
    return pd.concat([french('ff5', freq), french('mom', freq), f3], axis=1).dropna()


# =============================================================================
# REGRESII
# =============================================================================
def ols_hac(y, X, lags=None, const=True):
    """OLS cu erori standard Newey-West (Bartlett). Intoarce (coef, se, t, resid, r2)."""
    y = np.asarray(y, float)
    X = np.asarray(X, float)
    if X.ndim == 1:
        X = X[:, None]
    if const:
        X = np.column_stack([np.ones(len(y)), X])
    n, k = X.shape
    if lags is None:
        lags = int(np.floor(4 * (n / 100) ** (2 / 9)))
    XtX_inv = np.linalg.inv(X.T @ X)
    b = XtX_inv @ X.T @ y
    e = y - X @ b
    u = X * e[:, None]
    S = u.T @ u
    for l in range(1, lags + 1):
        w = 1 - l / (lags + 1)
        G = u[l:].T @ u[:-l]
        S += w * (G + G.T)
    V = XtX_inv @ S @ XtX_inv
    se = np.sqrt(np.diag(V))
    r2 = 1 - e.var() / y.var()
    return b, se, b / se, e, r2


def grs_test(excess, market_excess):
    """Testul Gibbons-Ross-Shanken (1989) pentru alfa = 0 pe N active, un factor.

    Sigma si varianta factorului sunt estimatorii de verosimilitate maxima (impartire la T),
    ca in lucrare; sub reziduuri normale i.i.d., W ~ F(N, T - N - 1) exact.
    """
    from scipy import stats
    R = np.asarray(excess, float)
    f = np.asarray(market_excess, float)
    T, N = R.shape
    X = np.column_stack([np.ones(T), f])
    B = np.linalg.lstsq(X, R, rcond=None)[0]
    alpha = B[0]
    E = R - X @ B
    Sigma = E.T @ E / T
    mu, s2 = f.mean(), f.var(ddof=0)
    stat = (T - N - 1) / N * (alpha @ np.linalg.solve(Sigma, alpha)) / (1 + mu ** 2 / s2)
    p = 1 - stats.f.cdf(stat, N, T - N - 1)
    return stat, p, alpha, B[1]

## Style and helpers

In [ ]:
# Stil standard MFM (identic cu SFM): transparent + ENG + legenda jos
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Culori brand
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Crimson  = '#DC3545'
Gray     = '#7F7F7F'
LightGray = '#DADADA'
PALETTE = [MainBlue, IDAred, Forest, Amber, Orange, Purple, Crimson, '#795548', '#17A2B8', '#6F42C1', '#20C997']

# modelele: FF3 si Carhart folosesc SMB-ul publicat al modelului cu trei factori; FF5 pe cel din fisierul cu cinci
FF3 = ['Mkt-RF', 'SMB_FF3', 'HML']
CARHART = FF3 + ['MOM']
FF5 = ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']
FF6 = FF5 + ['MOM']

SEED = 42
MAX_ABS_RET = 0.5        # prag pentru erori de date in seriile BVB (log-randament zilnic)
END = '2026-07-31'       # ultima luna a esantionului lunar folosit in capitol (T = 757 luni din iulie 1963)


def save_fig(name):
    """Salveaza figura ca PDF si PNG transparent, in folderul curent."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Plaseaza legenda in afara graficului, jos-centru."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def monthly_excess(symbols, start='1999-01-01'):
    """Randamente lunare simple in exces fata de RF (T-bill la o luna), pe lunile comune."""
    p = prices(symbols).resample('ME').last()
    r = p.pct_change().dropna()
    F = factors('M')
    idx = r.index.intersection(F.index)
    r = r.loc[idx].loc[start:END]
    F = F.loc[r.index]
    return r.sub(F['RF'], axis=0), F


def daily_excess_one(symbol, start=None):
    """Randamentul zilnic in exces al unui singur activ, pe calendarul factorilor.

    Se pastreaza doar zilele in care exista si pretul din ziua de tranzactionare precedenta
    (altfel randamentul ar acoperi mai multe zile, iar factorii doar una).
    """
    s = price(symbol)
    F = factors('D').loc[:END]                         # acelasi sfarsit de esantion ca datele lunare
    s = s.loc[s.index.isin(F.index)]
    cal = pd.Series(np.arange(len(F)), index=F.index)
    pos = cal.loc[s.index].values
    r = s.pct_change()
    consecutive = np.r_[False, np.diff(pos) == 1]
    r = r[consecutive].dropna()
    if start:
        r = r.loc[start:]
    Fx = F.loc[r.index]
    return (r - Fx['RF']).rename(symbol), Fx


def daily_excess(symbols, start=None):
    """Compatibilitate: dictionar {simbol: (exces, factori)} pe calendarul fiecarui activ."""
    return {s: daily_excess_one(s, start) for s in symbols}

BVB = ['TLV', 'SNP', 'BRD', 'TGN', 'FP', 'SNG', 'H2O', 'SNN', 'EL', 'DIGI', 'TEL']

## Analysis

In [ ]:
def part_c(return_series=False):
    """Momentum 12-1 si beta scazut pe blue chips BVB, doar cu informatie disponibila la formarea portofoliului.

    * lunile incomplete (ultima luna, daca datele se opresc inainte de sfarsitul ei) se elimina;
    * |r| > 50% intr-o luna = eveniment de capital neajustat (FP, septembrie 2023): randamentul este indisponibil;
      actiunea nu intra in clasamentele al caror semnal contine luna respectiva, iar in luna de detinere
      iese din media portofoliului (regula fixata dinainte);
    * eligibilitatea in luna t foloseste doar informatie de la sfarsitul lunii t-1 (semnal si pret disponibile).
    """
    names = [s for s in BVB if s not in ('H2O',)]
    px = pd.concat([price(s + '.RO') for s in names] + [price('BET')], axis=1)
    m = px.resample('ME').last().loc['2015-12-31':]
    last = px.index[-1]
    if (last + pd.offsets.BDay(1)).month == last.month:          # ultima luna nu e completa
        m = m.iloc[:-1]
    stocks = [s + '.RO' for s in names]
    raw = m.pct_change()
    flag = raw.abs() > 0.5
    rets = raw.mask(flag)                                         # randamente valide (NaN = eroare sau lipsa)
    gross = 1 + rets[stocks]                                      # randament indisponibil -> semnal indisponibil
    mom = gross.rolling(11, min_periods=11).apply(np.prod, raw=True).shift(2) - 1   # lunile t-12 ... t-2
    hold = rets[stocks].fillna(0.0).where(m[stocks].shift(1).notna()).mask(flag[stocks])   # randamentul lunii t
    rows = []
    for t in range(13, len(m)):
        date = m.index[t]
        avail = [s for s in stocks if pd.notna(mom[s].iloc[t]) and pd.notna(m[s].iloc[t - 1])]
        if len(avail) < 6:
            continue
        ms = mom.iloc[t][avail].sort_values()
        k = 3
        mom_ls = hold.iloc[t][ms.index[-k:]].mean() - hold.iloc[t][ms.index[:k]].mean()
        win = rets.iloc[max(0, t - 36):t]                         # doar lunile t-36 ... t-1
        betas = {}
        for s in avail:
            d = win[[s, 'BET']].dropna()
            if len(d) >= 18:
                betas[s] = np.cov(d[s], d['BET'])[0, 1] / d['BET'].var()
        if len(betas) >= 6:
            bs = pd.Series(betas).sort_values()
            lowb = hold.iloc[t][bs.index[:k]].mean() - hold.iloc[t][bs.index[-k:]].mean()
        else:
            lowb = np.nan
        rows.append((date, mom_ls, lowb, len(avail)))
    d = pd.DataFrame(rows, columns=['date', 'mom', 'lowbeta', 'n']).set_index('date')
    out = {}
    for c in ['mom', 'lowbeta']:
        x = d[c].dropna()
        b, se, t, _, _ = ols_hac(x.values, np.zeros((len(x), 0)), lags=3)
        out[c] = dict(T=len(x), mean_ann=x.mean() * 12, t_nw=t[0], sd_ann=x.std() * np.sqrt(12))
    out['start'] = str(d.index[0].date())
    out['end'] = str(d.index[-1].date())
    out['n_median'] = float(d['n'].median())
    out['n_flagged'] = int(flag[stocks].sum().sum())
    # C2: raspunsul AI pe acelasi esantion (semnal cu luna t inclusa, randamente log) si fiecare eroare separat
    look = (1 + rets[stocks]).rolling(12, min_periods=12).apply(np.prod, raw=True) - 1
    logh = np.log1p(hold)
    var, ser = {}, {'corrected': d['mom']}
    for lab, sig, h in [('lookahead', look, hold), ('log', mom, logh), ('ai_answer', look, logh)]:
        v = []
        for t in d.index:
            avail = [s for s in stocks if pd.notna(sig.loc[t, s]) and pd.notna(h.loc[t, s])]
            ss = sig.loc[t, avail].sort_values()
            v.append(h.loc[t, ss.index[-3:]].mean() - h.loc[t, ss.index[:3]].mean())
        v = pd.Series(v, index=d.index)
        ser[lab] = v
        var[lab] = dict(mean_ann=v.mean() * 12, sd_ann=v.std() * np.sqrt(12),
                        t_nw=ols_hac(v.values, np.zeros((len(v), 0)), lags=3)[2][0])
    out['c2_variants'] = var
    # C1: exemplu de clasament pentru ultima luna de detinere (semnal 12-1, eligibilitate la t-1)
    t = len(m) - 1
    avail = [s for s in stocks if pd.notna(mom[s].iloc[t]) and pd.notna(m[s].iloc[t - 1])]
    ms = mom.iloc[t][avail].sort_values(ascending=False)
    out['c1_example'] = dict(month=str(m.index[t].date()), signal={k[:-3]: float(v) for k, v in ms.items()},
                             hold={k[:-3]: float(hold.iloc[t][k]) for k in ms.index},
                             long=[k[:-3] for k in ms.index[:3]], short=[k[:-3] for k in ms.index[-3:]])
    fig_part_c(d)
    if return_series:
        return out, d, ser
    return out


def fig_part_c(d):
    """Evolutia a 1 leu in cele doua strategii long-short pe blue chips BVB."""
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(7.0, 3.0))
    for c, col, lab in [('mom', MainBlue, 'Momentum 12-1: top 3 minus bottom 3'),
                        ('lowbeta', IDAred, 'Low beta: lowest 3 minus highest 3 (36-month beta vs BET)')]:
        x = d[c].dropna()
        ax.plot(x.index, (1 + x).cumprod(), color=col, lw=1.0, label=lab)
    ax.axhline(1, color=Gray, lw=0.6, ls=':')
    ax.set_ylabel('Growth of 1 RON (long-short)')
    ax.set_title('Long-short sorts on BVB blue chips, monthly', fontsize=9, loc='left')
    legend_outside_bottom(ax, ncol=1)
    plt.tight_layout()
    save_fig('ch3_sem_bvb_factors')

## Run

In [ ]:
part_c()

![ch3_sem_bvb_factors](./ch3_sem_bvb_factors.png)

Output: `ch3_sem_bvb_factors.pdf`